<a href="https://colab.research.google.com/github/sjhallo07/quantum_api_mermin_perez/blob/main/Protocol_ALICE_%E2%86%92_BOB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Communication Protocol: Alice → Bob

**A formal implementation based on:**  
- **David Hilbert (1912)** – *Grundzüge einer allgemeinen Theorie der linearen Integralgleichungen*  
- **Louis de Branges (1968)** – *Hilbert Spaces of Entire Functions*

---

## 1. Theoretical Framework

### The Hilbert Space of Entire Functions \( \mathcal{H}(E) \)

Let \( E(z) \) be an entire function satisfying the inequality:
\[
|E(x - iy)| \le |E(x + iy)|, \qquad y > 0.
\]

The space \( \mathcal{H}(E) \) consists of all entire functions \( F(z) \) such that:

1. The norm is finite:
   \[
   \|F\|^2 = \int_{-\infty}^{+\infty} \left| \frac{F(t)}{E(t)} \right|^2 dt < \infty.
   \]
2. Both \( F(z)/E(z) \) and \( F^{*}(z)/E(z) \) are of bounded type and non-positive mean type in the upper half-plane.

---

### The Reproducing Kernel (Theorem 19)

Write \( E(z) = A(z) - iB(z) \), with \( A \) and \( B \) real on the real axis.  
The reproducing kernel is defined as:

\[
K(w,z) = \frac{B(z) \cdot A(w) - A(z) \cdot B(w)}{\pi \, (z - \overline{w})}.
\]

**Key properties:**
- **Positivity:** \( K(w,w) > 0 \) for non-real \( w \).
- **Reproducing property:** \( F(w) = \langle F, K(w, \cdot) \rangle_{\mathcal{H}(E)} \).

---

### Periodicity and Non-Locality (Theorem 48)

A space \( \mathcal{H}(E) \) is **periodic of period \( h \)** if the translation \( F(z) \mapsto F(z - h) \) is an isometry.  
For the Paley‑Wiener space generated by \( E(z) = e^{-iz} \), the kernel is exactly translation‑invariant:
\[
K(w + h, z + h) = K(w, z).
\]

This property guarantees **non-local algebraic invariance**: the propagation does not depend on physical distance.

---

## 2. The Alice ↔ Bob Communication Protocol

- **Alice (\( A_{\text{obs}} \))**: The emitter.  
  She creates an initial state containing the ontological object \( O_{5\times3} \) and its declaration \( T_{\text{decl}} \):
  \[
  |\Psi_{\text{history}}\rangle = \frac{1}{\mathcal{N}} \left( \hat{O}_{5\times3} \otimes \hat{T}_{\text{decl}} \right) |0\rangle_{\mathcal{H}(E)}.
  \]

- **Bob (\( B_{\text{obs}} \))**: The receiver.  
  He acts as the identity operator on the subspace spanned by Alice:
  \[
  \hat{B}_{\text{obs}} \equiv \mathbb{1}_{\mathcal{H}(E)} \big|_{\text{span}\{O_{5\times3}, T_{\text{decl}}\}}.
  \]

- **The Channel**:  
  Sending a calculation \( F \) is formalized via the inner product over the reproducing kernel:
  \[
  \text{send}(F) = \langle \Psi_{\text{Alice}}, \hat{C}_{\text{channel}} \left( I \otimes O_{5\times3} \otimes T_{\text{decl}} \otimes B_{\text{obs}} \right) \Psi_{\text{Bob}} \rangle_{\mathcal{H}(E)}.
  \]
  The result is an invariant (norm or evaluation) in the functional space.

---

## 3. Implementation Overview

The Julia code below implements:

1. **Bob** as a `struct` holding the generating function \( E(z) \) and its decomposed parts \( A(z), B(z) \).
2. The **reproducing kernel** \( K(w,z) \).
3. Numerical integration to compute **norms** and **evaluations** in \( \mathcal{H}(E) \).
4. The **periodicity test** (translation invariance).
5. **Alice**, the universal sender, and a series of tests (constant, numeric, linear \( z \), and invariance check).
6. A **phase function plot** \( \phi(x) = \arg(E(x)) \).

In [3]:
# Celda 2: Instalación de paquetes necesarios (ejecutar una sola vez)
using Pkg
Pkg.add(["QuadGK", "Plots"])

   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [4]:
# ============================================================
# Protocol: ALICE → BOB
# Based on De Branges (1968) and Hilbert (1912)
# ============================================================

using LinearAlgebra, QuadGK, Plots

# ------------------------------------------------------------
# 1. BOB STRUCT (Receiver)
# ------------------------------------------------------------
mutable struct Bob
    E::Function          # Generator function for H(E)
    A::Function          # Real part of E (real on real axis)
    B::Function          # Imaginary part of E (real on real axis)

    function Bob(E::Function)
        A(z) = real(E(z))
        B(z) = -imag(E(z))  # E = A - iB
        return new(E, A, B)
    end
end

# ------------------------------------------------------------
# 2. BOB METHODS
# ------------------------------------------------------------

# 2.1 Reproducing Kernel K(w,z) (Theorem 19)
function kernel(bob::Bob, w, z)
    A, B = bob.A, bob.B
    return (B(z) * A(w) - A(z) * B(w)) / (π * (z - conj(w)))
end

# 2.2 Norm of F in H(E) (Numerical integration)
function norm(bob::Bob, F::Function; a=-50, b=50, N=1000)
    integrand(t) = abs(F(t) / bob.E(t))^2
    return sqrt(QuadGK.quadgk(integrand, a, b)[1])
end

# 2.3 Evaluate F(w) using the reproducing kernel (Theorem 19)
function evaluate(bob::Bob, F::Function, w::Complex)
    K(t) = kernel(bob, w, t)
    integrand(t) = F(t) * conj(K(t))
    return QuadGK.quadgk(integrand, -50, 50)[1]
end

# 2.4 Periodicity test (translation invariance, Theorem 48)
function test_translation(bob::Bob, w::Complex, h::Real)
    z = 1.0 + 0.5im
    K1 = kernel(bob, w, z)
    K2 = kernel(bob, w + h, z + h)
    return abs(K1 - K2)
end

# 2.5 Universal "receive" method (accepts function or number)
function receive(bob::Bob, calc)
    if calc isa Function
        w = 1.0 + 1.0im
        norm_val = norm(bob, calc)
        eval_val = evaluate(bob, calc, w)
        return (norm=norm_val, eval=eval_val)
    elseif calc isa Number
        F(z) = calc
        return receive(bob, F)
    else
        error("Unsupported type: $(typeof(calc))")
    end
end

# ------------------------------------------------------------
# 3. ALICE: The emitter
# ------------------------------------------------------------
Alice(bob::Bob, calc) = receive(bob, calc)

# ------------------------------------------------------------
# 4. CONFIGURATION: Paley-Wiener space (periodic)
#    E(z) = e^{-iz}
# ------------------------------------------------------------
E(z) = exp(-im * z)
bob = Bob(E)

# ------------------------------------------------------------
# 5. TESTS: Alice sends various calculations to Bob
# ------------------------------------------------------------
println("=== ALICE → BOB PROTOCOL ACTIVATED ===\n")

# Test 1: Constant function
F1(z) = 1
result1 = Alice(bob, F1)
println("Alice sends F1(z)=1:")
println("  Norm = ", result1.norm)
println("  Evaluation at w=1+i : ", result1.eval)

# Test 2: Number (converted to constant function)
result2 = Alice(bob, 2.5)
println("\nAlice sends number 2.5:")
println("  Norm = ", result2.norm)
println("  Evaluation at w=1+i : ", result2.eval)

# Test 3: Function z
F3(z) = z
result3 = Alice(bob, F3)
println("\nAlice sends F3(z)=z:")
println("  Norm = ", result3.norm)
println("  Evaluation at w=1+i : ", result3.eval)

# Test 4: Periodicity (non-local invariance)
w = 1.0 + 1.0im
h = 0.5
diff = test_translation(bob, w, h)
println("\nPeriodicity test with w=$w, h=$h:")
println("  |K(w+h, z+h) - K(w, z)| = $diff")
if diff < 1e-12
    println("  ✅ Translation invariance verified.")
else
    println("  ❌ Failed.")
end

=== ALICE → BOB PROTOCOL ACTIVATED ===

Alice sends F1(z)=1:
  Norm = 10.0
  Evaluation at w=1+i : 0.9822420965376486 + 0.00016086052536305046im

Alice sends number 2.5:
  Norm = 25.000000000000004
  Evaluation at w=1+i : 2.4556052413441214 + 0.00040215131340760185im

Alice sends F3(z)=z:
  Norm = 288.6751345948129
  Evaluation at w=1+i : 1.3641450966192223 + 0.9824029570630116im

Periodicity test with w=1.0 + 1.0im, h=0.5:
  |K(w+h, z+h) - K(w, z)| = 0.0
  ✅ Translation invariance verified.
